<span style="color:#246485;font-weight:700;font-size:32px"> 
Option Analysis & Scoring Pipeline
</span> <br>

#### **architected by --> Harold Alfred Haugen**
-----

#### **Overview:** 
This notebook runs the end-to-end options analysis and scoring pipeline; 
from raw API data capture, premium and risk data build and scoring to a final 
scored universe structured for dashboard visualzation. 

The pipeline scans a universe of ~600 NYSE and Nasdaq symbols on a user-defined date, 
building a structured dataset of options premiums, implied volatility, historical volatility, 
price spike activity, and metrics across the forward term structure. 
All metrics are then scored and ranked into a two-dimensional premium/risk framework for put-selling alpha opportunities. 

#### **Objectives:** 
- Pull historical options chain and daily time-series price data from Alpha Vantage by symbol
- Build normalized put premium metrics across 4 DTE (Days to Expiration) windows 2-week, 1-month, next two future
  LEAPS (14, 30, over60_1, over60_2) and 4 delta-based strike buckets (ATM, Slight, Moderate, Far)
- Compute ATM straddle premium and three premium efficiency metrics
  (vs theoretical move, vs IV, vs realized vol)
- Compute Historical Volatility (HV_20, HV_30, HV_60) and IV/HV ratios per DTE window
- Compute spike analysis across 30 and 60-day windows with inward symbol and
  universal relative signals, producing a blended spike score based on frequency and
  magnitude
- Compute relative volatility vs SPY and QQQ benchmarks
- Score the universe on two axes — Premium Score and Risk Score,
  using weighted composites of the above metrics
- Assign each symbol to one of four quadrants (Q1 High Premium/Low Risk → Q4) for visualization
  and analysis
- Compute term structure slopes for premium and IV across DTE windows
- Output a single flat CSV per run for dashboard consumption

#### **Dataset Configurations:** 
- **Data source:**       Alpha Vantage Premium API (HISTORICAL_OPTIONS + TIME_SERIES_DAILY)
- **Option date:**       option_date  — the historical chain snapshot date (e.g. '2026-02-27')
- **HV as-of date:**     as_of_date   — price history for look-back analysis, truncated to prevent lookahead
- **Universe:**          ~600 symbols across NYSE and Nasdaq (ticker_list)
- **DTE windows:**       14-day (Friday-snapped), 30-day, over60_1, over60_2
- **Strike buckets:**    Delta based at: ATM (delta 0.40-0.60), Slight (0.25-0.40),
                         Moderate (0.15-0.25), Far (0.05-0.15)
- **Rate limit:**        75 calls/min by Alpha Vantage → batches of 37 symbols with 61s pause between batches
- **Benchmarks:**        SPY and QQQ HV_30 fetched once before the loop for relative vol

#### **Model Architecture Overview** 
The pipeline runs in four layers:

  1. API Layer (av_api_calls.py)
     Fetches options chain and daily prices per symbol. Manages rate limiting,
     error handling, and benchmark HV computation for SPY/QQQ.

  2. Premium Layer (option_prem_iv_builder.py)
     Parses and filters contracts by delta bucket and liquidity.
     Computes normalized premium (extrinsic/spot), aggregates by DTE/bucket,
     builds ATM straddle, and computes three premium efficiency metrics.

  3. Risk Layer (hist_vol_iv_risk_builder.py)
     Computes HV across three windows, extracts ATM IV per DTE window,
     builds IV/HV ratios with categorical signals, and runs spike analysis
     across 30 and 60-day windows with self-relative and universe signals.

  4. Scoring Layer (score_universe.py)
     Standardizes and weights premium and risk components into composite scores.
     Computes term structure regression slopes, premium efficiency signals,
     universe-relative spike and HV percentile ranks, and assigns quadrant labels.
     Outputs the final scored master DataFrame.

See ReadMe for further details on overall scoring model design. 

#### **Expected Results** 
- option_analysis_master  : flat DataFrame, one row per symbol, ~100+ columns
                            covering all premium, IV, HV, spike, and efficiency metrics.
                            ---> Pushes into Scoring Layer 
- scored_master           : option_analysis_master enriched with premium_score,
                            risk_score, quadrant, term structure slopes,
                            prem_efficiency_signal per DTE, blended spike_signal_universe,
                            HV_30_pct, relative_vol_spy_pct, relative_vol_qqq_pct
- error_log_df            : DataFrame of failed symbols with error message and
                            AV response detail for diagnosis
- CSV output              : scored_master saved to disk for dashboard consumption,
                            filename includes the option_date for run tracking

---

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
from IPython.display import clear_output
from typing import Optional
%config InlineBackend.figure_format = 'retina'
from scipy import stats

import requests
import json
import os
import dotenv
dotenv.load_dotenv()
import sys
import traceback
sys.tracebacklimit = 0 # turn off the error tracebacks

### reload amended source code / .py
%load_ext autoreload
%autoreload 2

In [2]:
os.getcwd()

'/Users/AlfHaugen/Python/Code/Markets_Project'

In [3]:
### Imports
import av_api_calls_III
import score_universe_IV

In [4]:
### symbols
# path_one = os.getcwd() + '/tickers_two_hun.txt'
path_one = os.getcwd() + '/tickers_six_hun.txt'
# path_one = os.getcwd() + '/tickers.txt'

with open(path_one, 'r') as f:
    ticker_list = [item.strip() for item in f.read().split(',')]

print(f' The number of symbols is {len(ticker_list)}')
ticker_list[-5:]

 The number of symbols is 593


['CM', 'ZION', 'CFG', 'MTB', 'FHN']

In [5]:
# Setting up request and params. 
API_KEY = os.getenv('MktPremiumKey')

In [6]:
#### Data Params:
option_date = '2026-03-13'   # date for historical options chain
as_of_date  = '2026-03-13'   # truncate HV history at this date

--------
### **Run the API / Option Scanner Data Build**

In [7]:
### Call Option Scanner -->
option_analysis_master, error_log_df = av_api_calls_III.option_analysis_scan(ticker_list, API_KEY, option_date, as_of_date)

Scanning 593 symbols
[593/593] Current: FHN
Succeeded : 476
Failed    : 116
API calls : 6
Elapsed   : 19:20
 Pushing in the spot value 21.74 
Building Option Premium Data
Building Historical Vol and Spike Data
Building Straddle Data
OK
------------------------------------------

Done: 477 succeeded, 116 failed
Total time: 19:21
Master shape: (477, 94)


In [8]:
print(option_analysis_master.shape)
option_analysis_master.head(5)

(477, 94)


,symbol,date,spot,expiration_14,premium_atm_14,iv_atm_14,premium_slight_14,iv_slight_14,premium_moderate_14,iv_moderate_14,premium_far_14,iv_far_14,expiration_30,premium_atm_30,iv_atm_30,premium_slight_30,iv_slight_30,premium_moderate_30,iv_moderate_30,premium_far_30,iv_far_30,expiration_over60_1,premium_atm_over60_1,iv_atm_over60_1,premium_slight_over60_1,iv_slight_over60_1,premium_moderate_over60_1,iv_moderate_over60_1,premium_far_over60_1,iv_far_over60_1,expiration_over60_2,premium_atm_over60_2,iv_atm_over60_2,premium_slight_over60_2,iv_slight_over60_2,premium_moderate_over60_2,iv_moderate_over60_2,premium_far_over60_2,iv_far_over60_2,straddle_14,put_atm_14,call_atm_14,prem_per_iv_primary_14,prem_per_iv_sec_14,prem_per_hv30_14,straddle_30,put_atm_30,call_atm_30,prem_per_iv_primary_30,prem_per_iv_sec_30,prem_per_hv30_30,straddle_over60_1,put_atm_over60_1,call_atm_over60_1,prem_per_iv_primary_over60_1,prem_per_iv_sec_over60_1,prem_per_hv30_over60_1,straddle_over60_2,put_atm_over60_2,call_atm_over60_2,prem_per_iv_primary_over60_2,prem_per_iv_sec_over60_2,prem_per_hv30_over60_2,HV_20,HV_30,HV_60,atm_iv_14,ratio_14,spread_14,signal_14,atm_iv_30,ratio_30,spread_30,signal_30,atm_iv_over60_1,ratio_over60_1,spread_over60_1,signal_over60_1,atm_iv_over60_2,ratio_over60_2,spread_over60_2,signal_over60_2,spike_count_30,spike_ratio_30,avg_spike_pct_30,max_spike_pct_30,spike_signal_30,spike_count_60,spike_ratio_60,avg_spike_pct_60,max_spike_pct_60,spike_signal_60,relative_vol_spy,relative_vol_qqq
0,AAPL,2026-03-13,250.12,2026-03-27,1.9784,30.4300,1.3054,33.3570,0.7976,36.1210,0.3985,41.3242,2026-04-10,2.6177,29.2920,2.0890,32.2190,1.1854,35.1455,0.4922,42.0722,2026-05-15,3.9334,31.0803,3.2459,34.4140,1.7092,38.8850,0.8212,43.8282,2026-06-18,4.4314,30.5115,4.1530,33.1945,2.2664,37.0965,1.0315,42.7877,4.19,1.98,2.21,0.5846,0.0650,0.0672,4.57,2.62,1.95,0.4681,0.0894,0.0889,6.61,3.93,2.68,0.4254,0.1266,0.1336,7.31,4.43,2.88,0.3864,0.1452,0.1505,0.2503,0.2944,0.2389,0.3043,1.034,0.0099,Equiv. Vol,0.2929,0.995,-0.0015,Equiv. Vol,0.3108,1.056,0.0164,Equiv. Vol,0.3051,1.036,0.0107,Equiv. Vol,1,0.73,5.13,5.13,Normal,5,1.83,3.80,5.13,Moderate,2.29,1.65
1,MSFT,2026-03-13,395.55,2026-03-27,1.8661,28.5605,1.3462,30.7552,0.7806,33.1945,0.3375,37.0967,2026-04-10,2.3891,28.0725,1.8203,30.2675,1.1187,32.7067,0.4736,37.0967,2026-05-15,4.2388,33.6823,3.3940,36.0234,1.9514,38.5600,0.8520,42.2457,2026-06-18,5.0030,32.9856,4.1324,34.9828,2.3120,37.3892,1.0482,40.7280,3.71,1.87,1.85,0.5514,0.0653,0.0664,4.66,2.39,2.27,0.4981,0.0851,0.0850,7.61,4.24,3.37,0.4519,0.1258,0.1508,8.71,5.00,3.71,0.4257,0.1517,0.1779,0.2233,0.2812,0.3240,0.2856,1.016,0.0044,Equiv. Vol,0.2807,0.998,-0.0004,Equiv. Vol,0.3368,1.198,0.0557,Equiv. Vol,0.3299,1.173,0.0487,Equiv. Vol,2,1.47,3.16,3.26,Normal,2,0.73,7.80,10.53,Normal,2.19,1.58
2,NVDA,2026-03-13,180.25,2026-03-27,2.9219,44.4137,1.8816,47.9910,1.1304,51.2425,0.5637,57.0962,2026-04-10,3.7517,40.9990,2.8779,44.4135,1.5576,48.3160,0.6359,55.6325,2026-05-15,5.3629,41.4867,4.5215,44.7390,2.3625,48.6410,0.9731,55.3400,2026-06-18,6.9443,43.9516,6.0823,46.5598,3.2302,50.2673,1.3024,56.6296,6.14,2.92,3.22,0.5865,0.0658,0.0707,6.56,3.75,2.81,0.4800,0.0915,0.0908,8.97,5.36,3.61,0.4326,0.1293,0.1298,11.97,6.94,5.02,0.4389,0.1580,0.1680,0.3617,0.4133,0.3559,0.4441,1.075,0.0308,Equiv. Vol,0.4100,0.992,-0.0033,Equiv. Vol,0.4149,1.004,0.0016,Equiv. Vol,0.4395,1.063,0.0262,Equiv. Vol,2,1.47,6.59,7.58,Normal,3,1.10,5.89,7.58,Normal,3.22,2.32
3,AMZN,2026-03-13,207.67,2026-03-27,2.3908,36.6087,1.6854,39.5355,1.0016,42.1370,0.4348,46.7550,2026-04-10,3.1769,35.1455,2.0826,38.5600,1.2664,40.5110,0.6254,46.3648,2026-05-15,4.9881,40.0235,4.1251,43.4380,2.3114,46.6900,1.0184,51.8280,2026-06-18,5.9602,38.0720,5.0160,40.5113,2.7899,43.9257,1.1248,49.1290,5.01,2.39,2.62,0.5811,0.0653,0.0755,6.81,3.18,3.63,0.5814,0.0904,0.1003,8.74,4.99,3.75,0.4369,0.1246,0.1574,10.70,5.96,4.74,0.4531,0.1566,0.1881,0.2545,0.3168,0.2877,0.3661,1.155,0.0493,Equiv. Vol,0.3515,1.

#### **Naming Convention**

In [10]:
file_size = '600_sym'
data_as_of = as_of_date
opt_file_name = 'opt_analysis_master_' + file_size + '_' + data_as_of + '.csv'
error_log_file_name = 'error_log_' + file_size + '_' + data_as_of + '.csv'
stats_log_file_name = 'stats_desc_' + file_size + '_' + data_as_of + '.csv'
field_error_file_name = 'field_errors_' + file_size + '_' + data_as_of + '.csv'
opt_file_name

'opt_analysis_master_600_sym_2026-03-13.csv'

-----
### **Error Log and DF Stats**
- Review Results logged during the API Loop Run

In [11]:
### reviewing the Errors logged
print(error_log_df.shape)
error_log_df.head(10)

(116, 3)


,symbol,error,av_info
0,BAC,"""['atm_iv_over60_1', 'ratio_over60_1', 'spread...","keys: ['Meta Data', 'Time Series (Daily)']"
1,TFC,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
2,TRV,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
3,NTRS,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
4,BEN,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
5,AMC,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
6,ICLN,"""None of [Index(['dte_window', 'atm_iv', 'hv_p...","keys: ['Meta Data', 'Time Series (Daily)']"
7,TBT,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
8,ITW,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"
9,AME,"""['atm_iv_14', 'ratio_14', 'spread_14', 'signa...","keys: ['Meta Data', 'Time Series (Daily)']"


In [12]:
### Save to CSV -->
error_log_df.to_csv(error_log_file_name, index=True)

#### **Statistics Review**

In [13]:
### Run Description Stats
data_describe = option_analysis_master.describe().T
# data_describe.head()

In [ ]:
### Save to CSV -->
data_describe.to_csv(stats_log_file_name, index=True)

#### **Field Specific Review**

In [ ]:
field_nan_report = av_api_calls_III.audit_non_numeric(option_analysis_master)
print(field_nan_report['symbol'].unique())
field_nan_report.head(5)

In [ ]:
### Save to CSV --> 
field_nan_report.to_csv(field_error_file_name, index=True)

----
### **Run block to Save Premium / Risk Data to CSV**

In [14]:
### Write to CSV
option_analysis_master.to_csv(opt_file_name, index=False)

<hr style="border:7px solid #246485">

## **Scoring Module**

In [ ]:
### If loading past data from saved CSV.....
# option_analysis_master = pd.read_csv('opt_analysis_master_200_sym_2026_02_27.csv')
# print(option_analysis_master.shape)
# option_analysis_master.head()

In [15]:
### Run data through Scoring Module: ----------->
scoring_data = score_universe_IV.score_universe(option_analysis_master)

In [17]:
print(scoring_data.shape)
scoring_data.sort_values('risk_score',ascending=False).head(4)

(477, 117)


,symbol,date,premium_score,risk_score,quadrant,spot,expiration_14,premium_atm_14,iv_atm_14,premium_slight_14,iv_slight_14,premium_moderate_14,iv_moderate_14,premium_far_14,iv_far_14,expiration_30,premium_atm_30,iv_atm_30,premium_slight_30,iv_slight_30,premium_moderate_30,iv_moderate_30,premium_far_30,iv_far_30,expiration_over60_1,premium_atm_over60_1,iv_atm_over60_1,premium_slight_over60_1,iv_slight_over60_1,premium_moderate_over60_1,iv_moderate_over60_1,premium_far_over60_1,iv_far_over60_1,expiration_over60_2,premium_atm_over60_2,iv_atm_over60_2,premium_slight_over60_2,iv_slight_over60_2,premium_moderate_over60_2,iv_moderate_over60_2,premium_far_over60_2,iv_far_over60_2,straddle_14,put_atm_14,call_atm_14,prem_per_iv_primary_14,prem_per_iv_sec_14,prem_per_hv30_14,straddle_30,put_atm_30,call_atm_30,prem_per_iv_primary_30,prem_per_iv_sec_30,prem_per_hv30_30,straddle_over60_1,put_atm_over60_1,call_atm_over60_1,prem_per_iv_primary_over60_1,prem_per_iv_sec_over60_1,prem_per_hv30_over60_1,straddle_over60_2,put_atm_over60_2,call_atm_over60_2,prem_per_iv_primary_over60_2,prem_per_iv_sec_over60_2,prem_per_hv30_over60_2,HV_20,HV_30,HV_60,atm_iv_14,ratio_14,spread_14,signal_14,atm_iv_30,ratio_30,spread_30,signal_30,atm_iv_over60_1,ratio_over60_1,spread_over60_1,signal_over60_1,atm_iv_over60_2,ratio_over60_2,spread_over60_2,signal_over60_2,spike_count_30,spike_ratio_30,avg_spike_pct_30,max_spike_pct_30,spike_signal_30,spike_count_60,spike_ratio_60,avg_spike_pct_60,max_spike_pct_60,spike_signal_60,relative_vol_spy,relative_vol_qqq,premium_slope,iv_slope,slope_divergence,wp_14,wp_30,wp_over60_1,wp_over60_2,premium_slope_pct,iv_slope_pct,slope_div_pct,prem_efficiency_signal_14,prem_efficiency_signal_30,prem_efficiency_signal_over60_1,prem_efficiency_signal_over60_2,spike_score_universe,spike_pct_universe,spike_signal_universe,HV_30_pct,relative_vol_spy_pct,relative_vol_qqq_pct
85,SOXS,2026-03-13,3.2650,6.0328,Q2 High Premium / High Risk,41.30,2026-03-27,9.5884,158.3621,7.3245,144.7906,3.0670,132.8667,1.6102,146.8500,2026-04-10,12.2821,147.3378,9.8931,133.6798,3.9952,143.9230,NaN,NaN,2026-05-15,18.0751,148.1182,17.0500,134.6010,7.0702,125.3870,3.4140,125.3870,2026-08-21,25.1412,149.1267,30.4404,139.7162,17.8874,131.2405,5.0121,120.5090,14.16,9.59,4.57,0.3793,0.0605,0.0108,17.37,12.28,5.08,0.3536,0.0834,0.0139,25.08,18.08,7.01,0.3387,0.1220,0.0204,28.41,25.14,3.27,0.2383,0.1686,0.0284,10.8008,8.8604,6.3216,1.5836,0.179,-7.2767,Discounted Vol,1.4734,0.166,-7.3870,Discounted Vol,1.4812,0.167,-7.3792,Discounted Vol,1.4913,0.168,-7.3691,Discounted Vol,1,0.73,303.36,303.36,Normal,1,0.37,303.36,303.36,Normal,69.01,49.67,0.218361,-0.000864,0.219226,9.1356,11.8043,17.8701,26.2010,1.000,0.155,1.000,Cheap + Thin,Cheap + Thin,Cheap + Thin,Cheap + Thin,3.5567,0.908,Extreme,1.000,1.000,1.000
467,ITUB,2026-03-13,-1.4952,2.8759,Q4 Low Premium / High Risk,7.98,2026-03-20,10.5054,269.7740,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-17,9.6178,114.1680,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-18,7.5815,39.5360,NaN,NaN,3.1328,42.4620,NaN,NaN,2026-09-18,8.6257,53.8443,5.6391,42.4620,NaN,NaN,NaN,NaN,12.45,10.51,1.94,0.2768,0.0389,0.2571,13.75,9.62,4.14,0.3232,0.0842,0.2354,15.48,7.58,7.89,0.6309,0.1918,0.1856,18.71,8.63,10.09,0.4013,0.1602,0.2111,0.3771,0.4086,0.3771,2.6977,6.603,2.2892,Very Rich Vol,1.1417,2.794,0.7331,Very Rich Vol,0.3954,0.968,-0.0132,Equiv. Vol,0.5384,1.318,0.1299,Rich Vol,1,0.73,5.33,5.33,Normal,2,0.73,5.34,5.36,Normal,3.18,2.29,NaN,-0.025487,NaN,NaN,NaN,NaN,8.0284,NaN,0.002,NaN,Rich + Thin,Rich + Thin,Cheap + Efficient,Rich + Thin,1.3481,0.210,Normal,0.539,0.538,0.539
82,AGQ,2026-03-13,3.6151,2.5736,Q2 High Premium / High Risk,138.14,2026-03-27,8.3938,131.7027,6.2824,133.0525,3.3842,142.9480,1.5202,159.7278,2026-04-10,11.3263,132.3422,9.9175,136.1187,5.0311,135.6310,NaN,NaN,2026-06-18,18.5329,136.4780,21.0155,129.0457,9.9175,124.3466,3.9931,128.6624,2026-09-18,20.5340,140.2461,31.0124,127.7721,15.5458,120.9972,7.5738,120.2655,14.16,8.39,5.77,0.4563,0.0637,0.0260,

#### **Save to CSV**

In [18]:
### Write to CSV -->
opt_file_name = 'option_scores_master' + '_' + file_size + '_' + data_as_of + '.csv'
scoring_data.to_csv(opt_file_name, index=False)

-----

### **End**
----